# AgentCore Identity OAuth 토큰 관리를 사용하는 Self-Hosted Agent

이 샘플에서는 [Amazon Bedrock AgentCore Identity](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/identity-getting-started.html)를 사용해 OAuth 2.0 토큰 흐름을 관리하는 self-hosted Python 에이전트를 구축합니다. OAuth 권한 부여 흐름, 토큰 저장소, 갱신 로직, secret 관리를 직접 구현하는 대신 AgentCore Identity가 모든 과정을 처리합니다.

## 개요

AgentCore Identity를 사용해 사용자를 대신하여 OAuth 2.0 access token을 얻는 로컬 Python 에이전트를 생성합니다. 에이전트는 OAuth provider에 직접 접근하지 않으며, AgentCore Identity가 내부에서 authorization code 교환, 토큰 저장, 갱신을 처리합니다.

### 튜토리얼 세부 정보

| 항목                | 세부 정보                                                                            |
|:--------------------|:-------------------------------------------------------------------------------------|
| 튜토리얼 유형       | 단계별                                                                               |
| 에이전트 유형       | 단일(self-hosted)                                                                    |
| Agentic Framework   | 없음(독립 실행형 Python)                                                             |
| 튜토리얼 구성 요소  | AgentCore Identity, Cognito User Pool, 로컬 callback server                         |
| 튜토리얼 분야       | 산업 공통                                                                            |
| 예제 난이도         | 초급                                                                                 |
| 사용 SDK            | boto3                                                                                |
| Credential Provider | 유형: OAuth2 - Custom provider(Cognito)                                              |

### 주요 기능

* custom OAuth2 credential provider를 사용하는 Outbound Auth
* workload identity 및 workload token 관리
* 로컬 callback server를 통한 OAuth2 session binding
* 에이전트 프레임워크 의존성 없이 boto3만 사용

## 아키텍처

<div style="text-align:center">
    <img src="images/self-hosted-agent-oauth.png" width="90%"/>
</div>

네 구성 요소 사이의 OAuth 흐름은 다음과 같습니다.

1. 에이전트가 자신과 사용자를 식별하여 AgentCore Identity에 **workload access token**을 요청합니다.
2. 에이전트가 OAuth2 토큰을 요청합니다. 캐시된 토큰이 없으므로 AgentCore Identity가 **authorization URL**과 **session URI**를 반환합니다.
3. 에이전트가 사용자의 브라우저에서 authorization URL을 엽니다.
4. 사용자가 OAuth server(Cognito)에서 인증하고 동의합니다.
5. OAuth server가 `session_id`와 함께 브라우저를 에이전트의 로컬 callback server(`http://127.0.0.1:8080/callback`)로 리디렉션합니다.
6. callback handler가 **`CompleteResourceTokenAuth`**를 호출하여 OAuth session을 사용자에게 바인딩합니다.
7. 에이전트가 AgentCore Identity에서 **access token**을 가져옵니다. OAuth code는 필요하지 않습니다.

## 주요 개념

- **Credential provider**: AgentCore Identity가 OAuth server와 통신하는 방법(discovery URL, client ID/secret)을 정의합니다.
- **Workload identity**: 에이전트를 나타냅니다. AgentCore는 에이전트가 OAuth 토큰을 요청할 때 사용하는 workload token을 발급합니다.
- **Session binding**: 사용자가 브라우저에서 권한을 부여한 후 앱이 `completeResourceTokenAuth`를 호출하여 OAuth session을 사용자에게 바인딩합니다. 이후 에이전트가 토큰을 가져올 수 있습니다.
- **Workload access token**: 에이전트 workload와 에이전트가 대신 작업하는 사용자를 모두 식별하는 단기 토큰입니다.
- **Session URI**: OAuth2 처리 중 여러 요청에 걸쳐 권한 부여 흐름의 상태를 추적합니다.

session binding에 대한 자세한 내용은 [OAuth 2.0 authorization URL session binding](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/oauth2-authorization-url-session-binding.html)을 참조하세요.

## 사전 요구 사항

시작하기 전에 다음 항목을 준비하세요.

- 적절한 권한(IAM, Cognito, Bedrock AgentCore 액세스)을 보유한 AWS 계정
- [Python 3.10+](https://www.python.org/downloads/)
- 설치된 [AWS CLI](https://docs.aws.amazon.com/cli/latest/userguide/getting-started-install.html) v2
- 설치된 [`jq`](https://jqlang.github.io/jq/download/)

AWS에 인증합니다.
```bash
# AWS IAM Identity Center(SSO)를 구성한 경우:
aws sso login

# AWS 콘솔 로그인을 사용하는 경우:
aws login
```

자격 증명이 정상적으로 작동하는지 확인합니다.
```bash
aws sts get-caller-identity
```

아직 구성하지 않았다면 리전을 설정합니다.
```bash
export AWS_REGION="us-east-1"
```

> **참고:** 이 가이드의 모든 명령은 `us-east-1`을 기준으로 합니다. 필요한 경우 원하는 리전으로 바꾸세요.

## SDK 및 종속성 설치

In [ ]:
!pip install -r requirements.txt --quiet

In [ ]:
from boto3.session import Session

boto_session = Session()
region = boto_session.region_name or "us-east-1"

print(f"Region: {region}")

identity_client = boto_session.client("bedrock-agentcore-control", region_name=region)
cognito_client = boto_session.client("cognito-idp", region_name=region)

## 1단계: Cognito User Pool 생성

이 빠른 시작에는 OAuth authorization server가 필요합니다. client ID, client secret, 테스트 사용자가 구성된 server가 이미 있다면 2단계로 이동하여 다음 셀의 `ISSUER_URL`, `CLIENT_ID`, `CLIENT_SECRET` 변수를 설정하세요.

준비된 server가 없다면 제공된 스크립트를 실행해 Amazon Cognito User Pool을 생성하세요.

> **참고:** 스크립트의 기본 리전은 `us-east-1`입니다. 다른 리전을 사용하려면 실행 전에 `export AWS_REGION="your-region"`을 설정하세요.

In [ ]:
!bash create_cognito.sh

위 스크립트가 출력한 값을 복사하여 아래 셀에 붙여 넣으세요.

In [ ]:
# create_cognito.sh 출력의 값을 붙여 넣기
USER_POOL_ID = ""  # 예: "us-east-1_aBcDeFgHi"
CLIENT_ID = ""  # 예: "1a2b3c4d5e6f7g8h9i0j"
CLIENT_SECRET = ""  # 예: "abcdef123456..."
ISSUER_URL = ""  # 예: "https://cognito-idp.us-east-1.amazonaws.com/<pool-id>/.well-known/openid-configuration"
COGNITO_USERNAME = ""  # 예: "AgentCoreTestUser1234"
COGNITO_PASSWORD = ""  # 예: "xYz...Aa1!"

## 2단계: Credential Provider 생성

credential provider는 AgentCore Identity가 에이전트를 대신해 OAuth authorization server와 상호 작용하는 방법을 정의합니다.

자체 authorization server를 사용하는 경우 `ISSUER_URL`, `CLIENT_ID`, `CLIENT_SECRET`을 해당 값으로 설정하세요. 1단계의 Cognito 스크립트를 사용했다면 위에서 이미 값이 설정되었습니다.

In [ ]:
import random
import string

CREDENTIAL_PROVIDER_NAME = "AgentCoreIdentityStandaloneProvider"

try:
    credential_provider_response = identity_client.create_oauth2_credential_provider(
        name=CREDENTIAL_PROVIDER_NAME,
        credentialProviderVendor="CustomOauth2",
        oauth2ProviderConfigInput={
            "customOauth2ProviderConfig": {
                "oauthDiscovery": {"discoveryUrl": ISSUER_URL},
                "clientId": CLIENT_ID,
                "clientSecret": CLIENT_SECRET,
            }
        },
    )
except identity_client.exceptions.ValidationException:
    # 이름이 이미 사용 중이면 임의의 접미사 추가
    suffix = "".join(random.choices(string.ascii_lowercase + string.digits, k=5))
    CREDENTIAL_PROVIDER_NAME = f"{CREDENTIAL_PROVIDER_NAME}-tutorial-{suffix}"
    print(f"Name taken, using: {CREDENTIAL_PROVIDER_NAME}")
    credential_provider_response = identity_client.create_oauth2_credential_provider(
        name=CREDENTIAL_PROVIDER_NAME,
        credentialProviderVendor="CustomOauth2",
        oauth2ProviderConfigInput={
            "customOauth2ProviderConfig": {
                "oauthDiscovery": {"discoveryUrl": ISSUER_URL},
                "clientId": CLIENT_ID,
                "clientSecret": CLIENT_SECRET,
            }
        },
    )

oauth2_callback_url = credential_provider_response["callbackUrl"]
print(f"Credential provider '{CREDENTIAL_PROVIDER_NAME}' created ✓")
print(f"OAuth2 Callback URL: {oauth2_callback_url}")

credential provider의 callback URL로 Cognito User Pool client를 업데이트하세요.

In [ ]:
cognito_client.update_user_pool_client(
    UserPoolId=USER_POOL_ID,
    ClientId=CLIENT_ID,
    CallbackURLs=[
        f"https://bedrock-agentcore.{region}.amazonaws.com/identities/oauth2/callback",
        oauth2_callback_url,
    ],
    AllowedOAuthFlows=["code"],
    AllowedOAuthScopes=["openid", "profile", "email"],
    AllowedOAuthFlowsUserPoolClient=True,
    SupportedIdentityProviders=["COGNITO"],
)
print("Cognito user pool client updated ✓")

## 3단계: Workload Identity 생성

workload identity는 토큰 요청 주체를 AgentCore Identity에 알려 줍니다. 로컬 개발에서는 `http://127.0.0.1:8080/callback`을 허용된 반환 URL로 등록하세요.

In [ ]:
WORKLOAD_NAME = "standalone-agent-identity"
CALLBACK_URL = "http://127.0.0.1:8080/callback"

try:
    identity_client.create_workload_identity(name=WORKLOAD_NAME)
    print(f"Workload identity '{WORKLOAD_NAME}' created.")
except identity_client.exceptions.ValidationException:
    # 이름이 이미 사용 중이면 임의의 접미사 추가
    suffix = "".join(random.choices(string.ascii_lowercase + string.digits, k=5))
    WORKLOAD_NAME = f"{WORKLOAD_NAME}-tutorial-{suffix}"
    print(f"Name taken, using: {WORKLOAD_NAME}")
    identity_client.create_workload_identity(name=WORKLOAD_NAME)
    print(f"Workload identity '{WORKLOAD_NAME}' created.")

identity_client.update_workload_identity(
    name=WORKLOAD_NAME,
    allowedResourceOauth2ReturnUrls=[CALLBACK_URL],
)
print(f"Workload identity updated with callback URL: {CALLBACK_URL} ✓")

## 4단계: 샘플 에이전트 생성

샘플 에이전트([`agent.py`](agent.py))에는 에이전트와 최소한의 웹 애플리케이션이 함께 포함되어 있습니다. 에이전트는 AgentCore Identity를 사용해 OAuth 2.0 grant를 시작하고, 웹 애플리케이션은 session binding callback을 처리합니다. 프로덕션에서는 사용자용 웹 앱(예: `https://myagentapp.com`)이 이 역할을 맡으며 일반적으로 에이전트와 분리됩니다.

로컬 개발에서는 웹 앱이 `http://127.0.0.1:8080`에서 수신 대기합니다.

## 5단계: 에이전트 실행

필수 환경 변수를 설정하고 실행하세요.

```bash
export CREDENTIAL_PROVIDER_NAME="AgentCoreIdentityStandaloneProvider"
export AWS_REGION="us-east-1"
export AGENT_USER_ID="quickstart-user"

python3 agent.py
```

> **참고:** 에이전트가 로컬 HTTP server를 시작하고 브라우저 창을 열기 때문에 이 Notebook 내부보다 터미널에서 실행하는 것이 좋습니다.

## 6단계: OAuth 흐름 테스트

1. 에이전트가 기본 브라우저에서 OAuth authorization URL을 자동으로 엽니다.
2. Cognito 테스트 사용자 자격 증명(1단계의 `COGNITO_USERNAME` / `COGNITO_PASSWORD`)으로 로그인합니다.
3. 브라우저가 `http://127.0.0.1:8080/callback`으로 리디렉션되고 "Authorization Complete!"가 표시됩니다.
4. 에이전트가 callback을 자동으로 감지하고 access token을 가져옵니다.

> **참고:** 브라우저가 자동으로 열리지 않으면 터미널에 출력된 URL을 복사하여 직접 여세요. 권한 부여를 완료하지 않고 에이전트를 중단했다면 다시 실행해 새 authorization URL을 받으세요.

### 예상 출력

```
============================================================
  AgentCore Identity - Local Agent
============================================================
[INFO]  App server listening on http://127.0.0.1:8080/callback
[INFO]  Workload identity 'standalone-agent-identity' exists - reusing.

  Opening your browser to authorize...

  https://bedrock-agentcore.us-east-1.amazonaws.com/identities/oauth2/authorize?...

  Waiting for you to complete authorization in the browser...
[INFO]  Session bound for session_id=ZGQxN2ZlYjEtODcy...
[INFO]  Authorization callback received.

============================================================
  Access token retrieved!

  Your agent now has consent to act on behalf of the user.
  AgentCore Identity handled the entire OAuth flow for you -
  no OAuth code required.
============================================================

  Token preview: eyJraWQiOiJxT0x0VGFhcVwiLCJhbGciOiJSUzI1...QuickStart

{
  "sub": "xxxxxxxx-xxxx-xxxx-xxxx-xxxxxxxxxxxx",
  "cognito:username": "AgentCoreTestUser1234",
  "token_use": "access",
  "auth_time": 1234567890,
  "exp": 1234571490
}

[INFO]  Done. The OAuth flow completed successfully.
```

## 정리

다음을 실행하여 이 튜토리얼에서 생성한 모든 리소스를 삭제하세요.

In [ ]:
# workload identity 삭제
identity_client.delete_workload_identity(name=WORKLOAD_NAME)
print("Workload identity deleted ✓")

# credential provider 삭제
identity_client.delete_oauth2_credential_provider(name=CREDENTIAL_PROVIDER_NAME)
print("Credential provider deleted ✓")

# User Pool을 삭제하기 전에 Cognito domain을 먼저 삭제
pool_info = cognito_client.describe_user_pool(UserPoolId=USER_POOL_ID)
domain = pool_info["UserPool"].get("Domain", "")
if domain:
    cognito_client.delete_user_pool_domain(UserPoolId=USER_POOL_ID, Domain=domain)
    print(f"Cognito domain '{domain}' deleted ✓")

# Cognito User Pool 삭제
cognito_client.delete_user_pool(UserPoolId=USER_POOL_ID)
print("Cognito user pool deleted ✓")

## 보안 모범 사례

1. 에이전트 코드에 **자격 증명을 하드코딩하지 마세요**.
2. AWS 자격 증명에는 **`aws sso login` 또는 환경 변수를 사용하세요**.
3. 특정 리소스로 범위를 제한한 **최소 권한 IAM 정책을 적용하세요**.
4. 프로덕션에서는 신뢰할 수 있는 CA가 발급한 **올바른 TLS 인증서를 사용하세요**.
5. **OAuth client secret을 정기적으로 교체하세요**.
6. CloudTrail을 통해 **액세스 로그를 감사하세요**.
7. **프로덕션에서는** 애플리케이션 server가 `CompleteResourceTokenAuth`를 호출하기 전에 사용자 session을 검증해야 합니다.

## 문제 해결

### `NoCredentialProviders` 또는 `ExpiredToken`
**원인**: AWS 자격 증명이 없거나 만료되었습니다.
**해결 방법**: `aws sso login` 또는 `aws login`을 다시 실행한 후 재시도하세요.

### workload identity의 `ResourceNotFoundException`
**원인**: workload identity가 삭제되었거나 생성되지 않았습니다.
**해결 방법**: 3단계 셀을 다시 실행해 생성하세요.

### 브라우저에 `error=redirect_mismatch` 표시
**원인**: Cognito callback URL에 AgentCore credential provider의 callback URL이 포함되어 있지 않습니다.
**해결 방법**: 2단계의 `update_user_pool_client` 셀을 다시 실행하세요.

### access token을 받지 못함
**원인**: 권한 부여를 완료하기 전에 브라우저를 닫았습니다.
**해결 방법**: `python3 agent.py`를 다시 실행해 새 authorization URL을 받으세요.

### 포트 8080이 이미 사용 중
**원인**: 다른 프로세스가 포트 8080을 사용 중입니다.
**해결 방법**: 해당 프로세스를 중지하거나 `export CALLBACK_PORT=9090`을 설정한 다음 workload identity의 허용된 반환 URL도 이에 맞게 업데이트하세요.

### `create-user-pool-domain`의 `InvalidParameterException`
**원인**: domain 이름이 이미 사용 중입니다.
**해결 방법**: 실행할 때마다 임의의 접미사를 생성하는 `create_cognito.sh` 스크립트를 다시 실행하세요.